# Paired Megaminx beam-capacity sweep — 1×T4

Public, self-contained Pilgrim benchmark notebook. It runs on one
visible Tesla T4, fixed Megaminx state and two real
checkpoint contracts: scalar `output_dim=1` and Q `output_dim=24` (the number
of generators). It does not convert, average, or otherwise substitute heads.

Methods:

- `pilgrim`: scalar Searcher for output-1 and QSearcher for output-24;
- `multigpu_beam_search`: the unchanged `WORLD_SIZE=1` production runner.

Every point is a fresh child process. A failure cannot poison later CUDA state.
After the first failure, the notebook estimates the memory boundary from the
last completed samples and runs exactly one confirmation point below it. It
does not finely bisect the boundary. `RUNNING` is never a result.


In [ ]:
from pathlib import Path

EXPECTED_GPU_TOKEN = 'T4'
BEAM_WIDTHS_BY_OUTPUT = {1: [6291456, 6553600]}
DEPTH_LIMIT = 8
BOUNDARY_ALIGNMENT = 2**20
BOUNDARY_SAFETY = 0.95

# Public attached bundle contract. It contains pinned source snapshots,
# the fixed p900 state/generators, provenance, and the two thin workers.
BUNDLE_GLOB = "gpu-beam-search-paper-1xt4-reviewer-notebooks"
EXPECTED_BUNDLE_MANIFEST_VERSION = 1
EXPECTED_GENERATORS = 24
EXPECTED_OUTPUT_DIMS = (1, 24)
METHODS = ('pilgrim',)
OUTPUT_DIMS = (1,)

WORK = Path("/kaggle/working/paper_paired_sweep")
RESULTS_JSONL = WORK / "results.jsonl"
SUMMARY_JSON = WORK / "summary.json"
RESULTS_CSV = WORK / "results.csv"


In [ ]:
import csv, hashlib, json, os, platform, shutil, subprocess, sys, time

# Hide Kaggle's second T4 before either worker imports torch.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
WORK.mkdir(parents=True, exist_ok=True)

gpu_rows = subprocess.check_output([
    "nvidia-smi", "--query-gpu=name,memory.total,driver_version",
    "--format=csv,noheader,nounits"], text=True).strip().splitlines()
if len(gpu_rows) < 1 or EXPECTED_GPU_TOKEN not in gpu_rows[0]:
    raise RuntimeError(f"requires 1x{EXPECTED_GPU_TOKEN}; physical rows={gpu_rows!r}")

bundle_matches = [p for p in Path("/kaggle/input").rglob("benchmark_manifest.json")
                  if BUNDLE_GLOB in str(p)]
if len(bundle_matches) != 1:
    raise RuntimeError(f"expected one public benchmark bundle manifest, got {bundle_matches}")
manifest = json.loads(bundle_matches[0].read_text(encoding="utf-8"))
dataset_root = bundle_matches[0].parent
payload_zip = dataset_root / "bundle_payload.zip"
if payload_zip.is_file():
    unpacked = WORK / "verified_bundle"
    if unpacked.exists():
        shutil.rmtree(unpacked)
    shutil.unpack_archive(payload_zip, unpacked)
    BUNDLE = unpacked
elif (dataset_root / "bundle_payload").is_dir():
    BUNDLE = dataset_root / "bundle_payload"
else:
    BUNDLE = dataset_root
if manifest.get("manifest_version") != EXPECTED_BUNDLE_MANIFEST_VERSION:
    raise RuntimeError("benchmark bundle manifest version mismatch")
if manifest.get("move_count") != EXPECTED_GENERATORS:
    raise RuntimeError("benchmark bundle is not the fixed 24-generator Megaminx contract")
if sorted(manifest.get("output_dims", [])) != list(EXPECTED_OUTPUT_DIMS):
    raise RuntimeError("both real output_dim=1 and output_dim=24 checkpoints are required")

# The manifest must prove two independently trained native heads. Merely naming
# files output1/output24 is insufficient for a reproducible measurement.
checkpoints = manifest.get("checkpoints", {})
for output_dim in EXPECTED_OUTPUT_DIMS:
    spec = checkpoints.get(str(output_dim), {})
    required = {"model_glob", "sha256", "architecture", "output_dim", "training_run_id", "native_head"}
    missing = sorted(required - set(spec))
    if missing:
        raise RuntimeError(f"checkpoint output_dim={output_dim} lacks provenance fields: {missing}")
    if int(spec["output_dim"]) != output_dim or spec["native_head"] is not True:
        raise RuntimeError(f"checkpoint is not a native output_dim={output_dim} head: {spec}")
    if checkpoints.get("1", {}).get("training_run_id") == checkpoints.get("24", {}).get("training_run_id"):
        raise RuntimeError("output-1 and output-24 must identify distinct native training runs")
    found = list(Path("/kaggle/input").rglob(Path(spec["model_glob"]).name))
    if len(found) > 1:
        raise RuntimeError(f"expected at most one attached model for output_dim={output_dim}: {found}")
    if found:
        checkpoint_path = found[0]
        actual = hashlib.sha256(checkpoint_path.read_bytes()).hexdigest()
        if actual != spec["sha256"]:
            raise RuntimeError(f"checkpoint SHA256 mismatch for output_dim={output_dim}")
    # Otherwise the pinned worker downloads the exact release asset and verifies the same SHA.

for rel, expected in manifest["sha256"].items():
    path = BUNDLE / rel
    actual = hashlib.sha256(path.read_bytes()).hexdigest()
    if actual != expected:
        raise RuntimeError(f"SHA256 mismatch: {rel}: {actual} != {expected}")

hardware = {
    "gpu_visible": gpu_rows[0], "physical_gpu_rows": gpu_rows,
    "cuda_visible_devices": os.environ["CUDA_VISIBLE_DEVICES"],
    "python": sys.version, "platform": platform.platform(),
    "bundle_manifest": manifest,
}
(WORK / "hardware.json").write_text(json.dumps(hardware, indent=2), encoding="utf-8")
print(json.dumps(hardware, indent=2), flush=True)


In [ ]:
def classify(returncode, timed_out, result_path, log_text):
    if timed_out:
        return "timeout"
    if result_path.is_file():
        payload = json.loads(result_path.read_text(encoding="utf-8"))
        if payload.get("status"):
            return payload["status"]
    low = log_text.lower()
    if "cuda out of memory" in low or "cuda_oom" in low:
        return "cuda_oom"
    if returncode in (-9, 137) or "sigkill" in low:
        return "host_oom_or_sigkill"
    return "completed" if returncode == 0 else "process_exit"


def run_point(method, output_dim, beam, attempt):
    tag = f"{method}_out{output_dim}_beam{beam}_attempt{attempt}"
    point_dir = WORK / tag
    point_dir.mkdir(parents=True, exist_ok=True)
    result_path = point_dir / "result.json"
    log_path = point_dir / "combined.log"
    worker = BUNDLE / "workers" / f"{method}.py"
    cmd = [sys.executable, str(worker), "--bundle", str(BUNDLE),
           "--output-dim", str(output_dim), "--beam", str(beam),
           "--depth", str(DEPTH_LIMIT), "--output", str(result_path)]
    started = time.time()
    timed_out = False
    proc = subprocess.run(cmd, text=True, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, env=os.environ.copy())
    returncode, log_text = proc.returncode, proc.stdout
    wall = time.time() - started
    log_path.write_text(log_text, encoding="utf-8", errors="replace")
    status = classify(returncode, timed_out, result_path, log_text)
    payload = json.loads(result_path.read_text(encoding="utf-8")) if result_path.is_file() else {}
    row = {
        "method": method, "output_dim": output_dim, "beam_width": beam,
        "attempt": attempt, "status": status, "returncode": returncode,
        "orchestrator_wall_s": wall, "log_path": str(log_path), **payload,
    }
    expected_checkpoint_sha = checkpoints[str(output_dim)]["sha256"]
    # A hard timeout can kill the child before it writes result.json. The
    # checkpoint and native-head contract were already verified in preflight;
    # preserve that provenance without pretending the worker attested it.
    if timed_out and not payload:
        row.update({"checkpoint_sha256": expected_checkpoint_sha,
                    "observed_output_dim": output_dim,
                    "move_count": EXPECTED_GENERATORS,
                    "attestation_source": "orchestrator_preflight"})
    if row.get("checkpoint_sha256") != expected_checkpoint_sha:
        raise RuntimeError(f"worker did not attest the selected native checkpoint: {row}")
    if int(row.get("observed_output_dim", -1)) != output_dim:
        raise RuntimeError(f"worker observed a non-native head shape: {row}")
    if int(row.get("move_count", -1)) != EXPECTED_GENERATORS:
        raise RuntimeError(f"worker changed the generator contract: {row}")
    with RESULTS_JSONL.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(row, sort_keys=True) + "\n")
    print("POINT_RESULT " + json.dumps(row, sort_keys=True), flush=True)
    return row


def confirmation_width(completed, failure_width):
    usable = [r for r in completed if r.get("peak_device_used_mib") is not None]
    estimate = float(failure_width)
    if len(usable) >= 2:
        left, right = usable[-2], usable[-1]
        db = right["beam_width"] - left["beam_width"]
        dm = right["peak_device_used_mib"] - left["peak_device_used_mib"]
        if db > 0 and dm > 0:
            slope = dm / db
            total_mib = float(str(hardware["gpu_visible"]).split(",")[1].strip())
            predicted = right["beam_width"] + (0.97 * total_mib - right["peak_device_used_mib"]) / slope
            estimate = min(float(failure_width), max(float(right["beam_width"]), predicted))
    candidate = int(estimate * BOUNDARY_SAFETY) // BOUNDARY_ALIGNMENT * BOUNDARY_ALIGNMENT
    last = max((r["beam_width"] for r in completed), default=0)
    return candidate if last < candidate < failure_width else None


if RESULTS_JSONL.exists():
    RESULTS_JSONL.unlink()
rows = []
for method in METHODS:
    for output_dim in OUTPUT_DIMS:
        first_failure = None
        completed = []
        for beam in BEAM_WIDTHS_BY_OUTPUT[output_dim]:
            row = run_point(method, output_dim, beam, 1)
            rows.append(row)
            if row["status"] == "completed":
                completed.append(row)
            else:
                first_failure = row
                break
        if first_failure is not None:
            confirm = confirmation_width(completed, first_failure["beam_width"])
            if confirm is not None:
                rows.append(run_point(method, output_dim, confirm, 1))


In [ ]:
keys = sorted({key for row in rows for key in row})
with RESULTS_CSV.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=keys, extrasaction="ignore")
    writer.writeheader(); writer.writerows(rows)

series = []
for method in METHODS:
    for output_dim in OUTPUT_DIMS:
        subset = [r for r in rows if r["method"] == method and r["output_dim"] == output_dim]
        completed = [r for r in subset if r["status"] == "completed" and r["attempt"] == 1]
        failed_widths = sorted({r["beam_width"] for r in subset if r["status"] != "completed"})
        series.append({
            "method": method, "output_dim": output_dim,
            "max_tested_complete": max((r["beam_width"] for r in completed), default=None),
            "first_failure": min(failed_widths, default=None),
            "boundary_policy": "two_broad_scalar_boundary_probes",
            "rows": len(subset),
        })
summary = {"hardware": hardware, "series": series, "rows": rows}
SUMMARY_JSON.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps({"series": series}, indent=2), flush=True)
if any(item["max_tested_complete"] is None for item in series):
    raise RuntimeError("benchmark invalid: every method/head series must complete its minimum beam")

try:
    import pandas as pd
    display(pd.DataFrame(rows))
    display(pd.DataFrame(series))
except Exception:
    pass
